# 03 — Synthetic Supplementary Analysis

Full learner grid, oracle-selected estimator results, per-estimator CDV advantage,
CDV support statistics, and tidy result DataFrames for further analysis.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'synthetic'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    build_full_learner_table, ranking_metrics_within_groups, ranking_comparison_table,
    resolve_alternative, describe_paired_test,
    METHODS_ORDER, LEARNERS_ORDER
)
from helpers.plotting import METHOD_LABELS, plot_scissors_chart

results_by_alpha = {}
for alpha in CFG.ALPHA_VALUES:
    full_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    debug_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    if os.path.exists(full_path):
        path = full_path
    else:
        path = debug_path
        print(f'WARNING: full-scale checkpoint not found for alpha={alpha:.2f}; '
              f'falling back to DEBUG checkpoint (small N_TRAIN/N_TEST, few seeds). '
              f'Run the full loop in 01_experiment.ipynb for real results.')
    results_by_alpha[alpha] = load_checkpoint(path)
    print(f'alpha={alpha:.2f}: {len(results_by_alpha[alpha])} seeds  [{os.path.basename(path)}]')


## 1. Full Learner Grid per Alpha — ATE MSE

In [ ]:
for alpha in CFG.ALPHA_VALUES:
    print(f'\nATE MSE — alpha={alpha:.2f}')
    table = build_full_learner_table(results_by_alpha[alpha], metric='ate_mse')
    display(table.round(5))

## 2. Full Learner Grid per Alpha — CATE MSE

In [ ]:
for alpha in CFG.ALPHA_VALUES:
    print(f'\nCATE MSE — alpha={alpha:.2f}')
    table = build_full_learner_table(results_by_alpha[alpha], metric='cate_mse')
    display(table.round(5))

## 3. Oracle-Selected Estimator Results

In [ ]:
oracle_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for seed, sr in res.items():
        for method in METHODS_ORDER:
            oracle = sr.get('oracle', {}).get(method, {})
            m = oracle.get('metrics', {})
            oracle_rows.append({
                'alpha': alpha, 'outer_seed': seed, 'method': method,
                'selected_learner': oracle.get('selected_learner', 'N/A'),
                'ate_mse': m.get('ate_mse', np.nan),
                'cate_mse': m.get('cate_mse', np.nan),
            })

oracle_df = pd.DataFrame(oracle_rows)
print('Oracle ATE MSE by alpha and method:')
display(oracle_df.groupby(['alpha', 'method'])['ate_mse'].mean().unstack().round(5))

print('\nOracle learner selection frequency per alpha and method:')
display(oracle_df.groupby(['alpha', 'method', 'selected_learner']).size().unstack(fill_value=0))

oracle_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'oracle_results.parquet'), index=False)

## 4. Ranking Metrics — Kendall τ and Spearman ρ (primary learner) per Alpha

Undefined (NaN) at alpha=0, where true ITE is constant and rank correlation is not identifiable.


In [ ]:
print(f'Ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds, per alpha')
ranking_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        tau_vals, rho_vals = [], []
        for sr in res.values():
            m = sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {})
            tau_vals.append(m.get('kendall_tau', np.nan))
            rho_vals.append(m.get('spearman_rho', np.nan))
        ranking_rows.append({
            'alpha': alpha,
            'Method': METHOD_LABELS.get(method, method),
            'Kendall τ mean': np.nanmean(tau_vals),
            'Kendall τ std':  np.nanstd(tau_vals),
            'Spearman ρ mean': np.nanmean(rho_vals),
            'Spearman ρ std':  np.nanstd(rho_vals),
            'N seeds': int(np.sum(np.isfinite(tau_vals))),
        })

ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df.round(4))
ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics.csv'), index=False)


In [ ]:
selected_learner_rank = CFG.PRIMARY_LEARNER  # change to compare a different learner

print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
print(describe_paired_test('CDV_SEPARATE', 'method', 'Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    print(f'\nCDV_SEPARATE vs other methods — Kendall τ ({selected_learner_rank}), alpha={alpha:.2f}')
    tau_cmp = ranking_comparison_table(res, selected_learner_rank, metric='kendall_tau',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    tau_cmp.index = [METHOD_LABELS.get(m, m) for m in tau_cmp.index]
    display(tau_cmp.round(4))

    print(f'CDV_SEPARATE vs other methods — Spearman ρ ({selected_learner_rank}), alpha={alpha:.2f}')
    rho_cmp = ranking_comparison_table(res, selected_learner_rank, metric='spearman_rho',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    rho_cmp.index = [METHOD_LABELS.get(m, m) for m in rho_cmp.index]
    display(rho_cmp.round(4))


## 5. Within-CDV Ranking Metrics — Kendall τ and Spearman ρ (primary learner) per Alpha

The Section 4 metrics pool predictions across all CDV groups (and OTHER) before ranking.
Here, Kendall τ / Spearman ρ are computed separately WITHIN each CDV group, then
size-weighted averaged across groups — isolating within-subgroup ranking quality from
across-group ordering effects.


In [ ]:
print(f'Within-CDV ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds, per alpha')
within_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        tau_vals, rho_vals = [], []
        for sr in res.values():
            pred = sr.get('predictions', {}).get(method, {}).get(CFG.PRIMARY_LEARNER, {})
            ite_pred = np.asarray(pred.get('ite_pred', []))
            ite_true = np.asarray(pred.get('ite_true', []))
            variant = np.asarray(pred.get('variant', []))
            if len(ite_pred) == 0:
                continue
            m = ranking_metrics_within_groups(ite_pred, ite_true, variant)
            tau_vals.append(m['kendall_tau_within'])
            rho_vals.append(m['spearman_rho_within'])
        within_rows.append({
            'alpha': alpha,
            'Method': METHOD_LABELS.get(method, method),
            'Kendall τ (within-CDV) mean': np.nanmean(tau_vals),
            'Kendall τ (within-CDV) std':  np.nanstd(tau_vals),
            'Spearman ρ (within-CDV) mean': np.nanmean(rho_vals),
            'Spearman ρ (within-CDV) std':  np.nanstd(rho_vals),
            'N seeds': int(np.sum(np.isfinite(tau_vals))),
        })

within_ranking_df = pd.DataFrame(within_rows)
display(within_ranking_df.round(4))
within_ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics_within_cdv.csv'), index=False)


## 6. Per-Estimator CDV Advantage per Alpha

In [ ]:
advantage_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for learner in LEARNERS_ORDER:
        cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        gs_vals  = [sr['metrics'].get('GLOBAL_SENTINEL', {}).get(learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        cdv_mean = np.nanmean(cdv_vals)
        gs_mean  = np.nanmean(gs_vals)
        improvement_pct = (gs_mean - cdv_mean) / gs_mean * 100 if gs_mean > 0 else np.nan
        advantage_rows.append({
            'alpha': alpha, 'learner': learner,
            'cdv_ate_mse': cdv_mean, 'global_ate_mse': gs_mean,
            'improvement_pct': improvement_pct,
        })

adv_df = pd.DataFrame(advantage_rows)
print('CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha:')
display(adv_df.pivot(index='learner', columns='alpha', values='improvement_pct').round(2))

In [ ]:
selected_learner = CFG.PRIMARY_LEARNER  # change to compare a different learner
baseline_methods = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']

vs_methods_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(selected_learner, {}).get('ate_mse', np.nan) for sr in res.values()]
    cdv_mean = np.nanmean(cdv_vals)
    for method in baseline_methods:
        base_vals = [sr['metrics'].get(method, {}).get(selected_learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        base_mean = np.nanmean(base_vals)
        improvement_pct = (base_mean - cdv_mean) / base_mean * 100 if base_mean > 0 else np.nan
        vs_methods_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'cdv_ate_mse': cdv_mean, 'baseline_ate_mse': base_mean,
            'improvement_pct': improvement_pct,
        })

vs_methods_df = pd.DataFrame(vs_methods_rows)
print(f'CDV_SEPARATE improvement (%) over each method (learner={selected_learner}, ATE MSE):')
display(vs_methods_df.pivot(index='method', columns='alpha', values='improvement_pct').round(2))

## 7. Per-Estimator CDV Advantage per Alpha — CATE

In [ ]:
advantage_cate_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for learner in LEARNERS_ORDER:
        cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(learner, {}).get('cate_mse', np.nan) for sr in res.values()]
        gs_vals  = [sr['metrics'].get('GLOBAL_SENTINEL', {}).get(learner, {}).get('cate_mse', np.nan) for sr in res.values()]
        cdv_mean = np.nanmean(cdv_vals)
        gs_mean  = np.nanmean(gs_vals)
        improvement_pct = (gs_mean - cdv_mean) / gs_mean * 100 if gs_mean > 0 else np.nan
        advantage_cate_rows.append({
            'alpha': alpha, 'learner': learner,
            'cdv_cate_mse': cdv_mean, 'global_cate_mse': gs_mean,
            'improvement_pct': improvement_pct,
        })

adv_cate_df = pd.DataFrame(advantage_cate_rows)
print('CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha (CATE MSE):')
display(adv_cate_df.pivot(index='learner', columns='alpha', values='improvement_pct').round(2))

In [ ]:
selected_learner_cate = CFG.PRIMARY_LEARNER  # change to compare a different learner
baseline_methods_cate = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']

vs_methods_cate_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(selected_learner_cate, {}).get('cate_mse', np.nan) for sr in res.values()]
    cdv_mean = np.nanmean(cdv_vals)
    for method in baseline_methods_cate:
        base_vals = [sr['metrics'].get(method, {}).get(selected_learner_cate, {}).get('cate_mse', np.nan) for sr in res.values()]
        base_mean = np.nanmean(base_vals)
        improvement_pct = (base_mean - cdv_mean) / base_mean * 100 if base_mean > 0 else np.nan
        vs_methods_cate_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'cdv_cate_mse': cdv_mean, 'baseline_cate_mse': base_mean,
            'improvement_pct': improvement_pct,
        })

vs_methods_cate_df = pd.DataFrame(vs_methods_cate_rows)
print(f'CDV_SEPARATE improvement (%) over each method (learner={selected_learner_cate}, CATE MSE):')
display(vs_methods_cate_df.pivot(index='method', columns='alpha', values='improvement_pct').round(2))

## 8. Build Tidy Main Results DataFrame

In [ ]:
main_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for seed, sr in res.items():
        for method in METHODS_ORDER:
            for learner in LEARNERS_ORDER:
                m = sr.get('metrics', {}).get(method, {}).get(learner, {})
                main_rows.append({
                    'dataset': 'synthetic',
                    'outer_seed': seed, 'alpha': alpha,
                    'method': method, 'learner': learner,
                    'ate_mse': m.get('ate_mse', np.nan),
                    'cate_mse': m.get('cate_mse', np.nan),
                    'kendall_tau': m.get('kendall_tau', np.nan),
                    'spearman_rho': m.get('spearman_rho', np.nan),
                    'n_train': sr.get('n_train', np.nan),
                    'n_test': sr.get('n_test', np.nan),
                    'n_retained_cdvs': len(sr.get('retained_cdv_info', {})),
                    'pct_other_train': sr.get('pct_other_train', np.nan),
                })

main_df = pd.DataFrame(main_rows)
out_path = os.path.join(CFG.ARTIFACTS_DIR, 'main_results.parquet')
main_df.to_parquet(out_path, index=False)
print(f'Tidy results: {main_df.shape}')
print(f'Saved: {out_path}')

## 9. CDV Support Statistics per Alpha

In [ ]:
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    pct_others = [sr.get('pct_other_train', np.nan) for sr in res.values()]
    n_retained = [len(sr.get('retained_cdv_info', {})) for sr in res.values()]
    print(f'alpha={alpha:.2f}: retained_CDVs={np.nanmean(n_retained):.1f}±{np.nanstd(n_retained):.1f}, '
          f'OTHER%={np.nanmean(pct_others):.1f}±{np.nanstd(pct_others):.1f}')